# Large Language Models

Generated from the book sources. Do not edit by hand: changes belong in the `.qmd` chapter.

> Setup the book does not print. Later cells depend on the state it creates, so run it.

In [ ]:
import os
os.environ["TRANSFORMERS_VERBOSITY"] = "error"
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"

sentence = (
    "This coffee is mind-blowingly good, "
    "but the aftertaste lingers."
)

# Naive split
# print("Naive split:", sentence.split(" "))

# Hugging Face: BERT (WordPiece)
from transformers import AutoTokenizer
bert_tok = AutoTokenizer.from_pretrained(
    "bert-base-uncased"
)
# print("BERT tokens:", bert_tok.tokenize(sentence))

# Hugging Face: GPT-2 (BPE)
gpt2_tok = AutoTokenizer.from_pretrained("gpt2")
# print("GPT-2 tokens:", gpt2_tok.tokenize(sentence))


def fmt(tokens):
    return " • ".join(tokens)

def md_tokens(tokens):
    # simple version; good enough unless you
    # have backticks inside tokens
    return " ".join(f"`{t}`" for t in tokens)

> Setup the book does not print. Later cells depend on the state it creates, so run it.

In [ ]:
from nltk.stem import WordNetLemmatizer

wnl = WordNetLemmatizer()

examples = [
    ("running", "v"),
    ("ran", "v"),
    ("runs", "v"),
    ("better", "a"),
    ("worse", "a"),
    ("studies", "v"),
    ("studying", "v"),
]

rows = []
for word, pos in examples:
    lemma = wnl.lemmatize(word, pos=pos)
    rows.append((word, pos, lemma))

lem_table = rows

> Setup the book does not print. Later cells depend on the state it creates, so run it.

In [ ]:
import os, numpy as np

GLOVE_TXT = "../assets/glove.6B.100d.txt"
if not os.path.exists(GLOVE_TXT):
    raise FileNotFoundError(
        f"GloVe file not found at {GLOVE_TXT}. "
        "Place glove.6B.100d.txt under ../assets/."
    )

# Parse the text file (word + 100 floats per line)
stoi, itos, vecs = {}, [], []
with open(GLOVE_TXT, "r", encoding="utf8") as f:
    for i, line in enumerate(f):
        parts = line.rstrip().split(" ")
        w = parts[0]
        nums = np.array(parts[1:], dtype=np.float32)
        stoi[w] = i
        itos.append(w)
        vecs.append(nums)

vectors = np.vstack(vecs)  # [V, 100]

def v(w):
    return vectors[stoi[w]]

def cos_sim(a, b):
    denom = np.linalg.norm(a) * np.linalg.norm(b)
    return float(np.dot(a, b) / denom)

> Setup the book does not print. Later cells depend on the state it creates, so run it.

In [ ]:
import numpy as np
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

labels = [
    "king","queen","man","woman","princess","monarch",
    "doctor","nurse","teacher","student",
    "dog","cat","tiger","lion",
    "apple","banana","grape","pizza","burger",
    "happy","sad","angry","joyful","depressed",
    "computer","laptop","keyboard","internet","software",
]
labels = [w for w in labels if w in stoi]

X = np.stack([v(w) for w in labels])
X2 = PCA(n_components=2, random_state=42).fit_transform(X)

fig, ax = plt.subplots(figsize=(6.5, 4.8))

# scatter points
ax.scatter(X2[:, 0], X2[:, 1], zorder=2)

# compute plot-dependent offsets so labels
# are not on top of points
xmin, xmax = X2[:, 0].min(), X2[:, 0].max()
ymin, ymax = X2[:, 1].min(), X2[:, 1].max()
dx_base = 0.02 * (xmax - xmin)
dy_base = 0.02 * (ymax - ymin)
cx, cy = X2.mean(axis=0)

for (x, y), w in zip(X2, labels):
    # push labels slightly away from the center of the cloud
    sx = np.sign(x - cx) or 1.0
    sy = np.sign(y - cy) or 1.0
    dx = sx * dx_base
    dy = sy * dy_base
    ax.text(
        x + dx,
        y + dy,
        w,
        fontsize=10,
        ha="center",
        va="center",
        zorder=3,
    )

# ----- axes with arrows -----

# get limits after plotting
xmin, xmax = ax.get_xlim()
ymin, ymax = ax.get_ylim()

# remove the standard frame
for spine in ax.spines.values():
    spine.set_visible(False)

# draw bottom x-axis arrow
ax.annotate(
    "",
    xy=(xmax, ymin),
    xytext=(xmin, ymin),
    arrowprops=dict(arrowstyle="->", lw=1.2),
    zorder=4,
)

# draw left y-axis arrow
ax.annotate(
    "",
    xy=(xmin, ymax),
    xytext=(xmin, ymin),
    arrowprops=dict(arrowstyle="->", lw=1.2),
    zorder=4,
)

# no tick labels (numbers not meaningful)
ax.set_xticks([])
ax.set_yticks([])

ax.grid(False)
fig.tight_layout()
plt.show()

> Setup the book does not print. Later cells depend on the state it creates, so run it.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

coords = {w: X2[i] for i, w in enumerate(labels)}
apple = coords["apple"]
banana = coords["banana"]
laptop = coords["laptop"]

alpha = 0.65
apple_food = apple + alpha * (banana - apple)
apple_tech = apple + alpha * (laptop - apple)

fig, ax = plt.subplots(figsize=(6.6, 4.8))

ax.scatter(
    X2[:, 0],
    X2[:, 1],
    alpha=0.18,
    zorder=1,
)

def mark(point, text, color):
    ax.scatter(
        [point[0]],
        [point[1]],
        s=70,
        color=color,
        zorder=3,
    )
    ax.text(
        point[0],
        point[1],
        text,
        fontsize=11,
        ha="left",
        va="bottom",
        zorder=4,
    )

mark(banana, "banana", "#2f5aa6")
mark(laptop, "laptop", "#2f5aa6")
mark(apple, "apple (static)", "#111111")
mark(apple_food, "apple (food)", "#1d4e89")
mark(apple_tech, "apple (tech)", "#557591")

ax.annotate(
    "",
    xy=(apple_food[0], apple_food[1]),
    xytext=(apple[0], apple[1]),
    arrowprops=dict(arrowstyle="->", lw=1.6),
    zorder=2,
)
ax.annotate(
    "",
    xy=(apple_tech[0], apple_tech[1]),
    xytext=(apple[0], apple[1]),
    arrowprops=dict(arrowstyle="->", lw=1.6),
    zorder=2,
)

for spine in ax.spines.values():
    spine.set_visible(False)
ax.set_xticks([])
ax.set_yticks([])

fig.tight_layout()
plt.show()

In [ ]:
print("king vs queen:", round(cos_sim(v("king"), v("queen")), 3))
print("king vs apple:", round(cos_sim(v("king"), v("apple")), 3))

> Setup the book does not print. Later cells depend on the state it creates, so run it.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
import warnings, logging

# suppress all matplotlib warnings + log messages
warnings.filterwarnings("ignore", module="matplotlib")
logging.getLogger('matplotlib').setLevel(logging.ERROR)

words4 = ["king", "man", "woman", "queen"]
words4 = [w for w in words4 if w in stoi]

X = np.stack([v(w) for w in words4])
X2 = PCA(n_components=2, random_state=42).fit_transform(X)
pt = {w: X2[i] for i, w in enumerate(words4)}

fig, ax = plt.subplots(figsize=(5, 4))

ax.scatter(X2[:,0], X2[:,1], zorder=3)

label_offset = 0.03
for w in words4:
    x, y = pt[w]
    ax.text(
        x + label_offset,
        y + label_offset,
        w,
        fontsize=11,
        ha="left",
        va="bottom",
        zorder=5,
    )

arrow_kw = dict(
    arrowstyle="->",
    lw=1.8,
    alpha=0.8,
    zorder=6,
)

def arrow(a, b, shrink=0.08):
    xa, ya = pt[a]
    xb, yb = pt[b]
    dx, dy = xb - xa, yb - ya
    ax.annotate(
        "",
        xy=(xa + dx*(1 - shrink), ya + dy*(1 - shrink)),
        xytext=(xa + dx*shrink, ya + dy*shrink),
        arrowprops=arrow_kw
    )

arrow("man", "woman")
arrow("king", "queen")

analogy_2d = pt["king"] - pt["man"] + pt["woman"]
ax.scatter([analogy_2d[0]], [analogy_2d[1]], zorder=4)
ax.text(
    analogy_2d[0] + label_offset,
    analogy_2d[1] + label_offset,
    "king - man + woman",
    fontsize=10,
    style="italic",
    ha="left",
    va="bottom",
    alpha=0.8,
    zorder=5
)

# --------- axis arrows ---------

ax.set_aspect("equal", adjustable="datalim")

# now expand limits *after* aspect enforcement
xmin, xmax = ax.get_xlim()
ymin, ymax = ax.get_ylim()
dx = 0.1 * (xmax - xmin)
dy = 0.1 * (ymax - ymin)
ax.set_xlim(xmin - dx, xmax + dx)
ax.set_ylim(ymin - dy, ymax + dy)

# remove frame
for spine in ax.spines.values():
    spine.set_visible(False)

# draw axis arrows
# enforce equal aspect first
ax.set_aspect("equal", adjustable="datalim")

# Let Matplotlib update limits based on aspect
plt.draw()

# Now safely read final limits
xmin, xmax = ax.get_xlim()
ymin, ymax = ax.get_ylim()

# Remove the box frame
for spine in ax.spines.values():
    spine.set_visible(False)

# Draw x-axis arrow
ax.annotate(
    "",
    xy=(xmax, ymin),
    xytext=(xmin, ymin),
    arrowprops=dict(arrowstyle="->", lw=1.2),
    zorder=10,
)

# Draw y-axis arrow
ax.annotate(
    "",
    xy=(xmin, ymax),
    xytext=(xmin, ymin),
    arrowprops=dict(arrowstyle="->", lw=1.2),
    zorder=10,
)

ax.set_xticks([])
ax.set_yticks([])

fig.tight_layout()
plt.show()

> Setup the book does not print. Later cells depend on the state it creates, so run it.

In [ ]:
import torch
import numpy as np
from transformers import AutoTokenizer, AutoModel

attn_tok = AutoTokenizer.from_pretrained("gpt2")
attn_model = AutoModel.from_pretrained(
    "gpt2", attn_implementation="eager"
)
attn_model.eval()

attn_sentence = "The bank raised interest rates today"
attn_inputs = attn_tok(attn_sentence, return_tensors="pt")
with torch.no_grad():
    attn_out = attn_model(
        **attn_inputs, output_attentions=True
    )

attn_tokens = [
    attn_tok.decode([i]).strip()
    for i in attn_inputs["input_ids"][0]
]
n_tok = len(attn_tokens)

> Setup the book does not print. Later cells depend on the state it creates, so run it.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

book_blue_cmap = LinearSegmentedColormap.from_list(
    "book_blue", ["#ffffff", "#1d4e89"]
)

def draw_attention(ax, A, tokens, annotate=True,
                   tick_fs=9, ann_fs=8.5):
    n = len(tokens)
    ax.imshow(A, cmap=book_blue_cmap, vmin=0, vmax=1)
    for i in range(n):
        for j in range(n):
            if j > i:
                ax.add_patch(plt.Rectangle(
                    (j - 0.5, i - 0.5), 1, 1,
                    color="#eef4fa", zorder=2,
                ))
            elif annotate:
                val = A[i, j]
                ax.text(
                    j, i, f"{val:.2f}",
                    ha="center", va="center",
                    fontsize=ann_fs, zorder=3,
                    color="#ffffff" if val > 0.5
                    else "#1d4e89",
                )
    ax.set_xticks(range(n))
    ax.set_xticklabels(tokens, rotation=45,
                       ha="right", fontsize=tick_fs)
    ax.set_yticks(range(n))
    ax.set_yticklabels(tokens, fontsize=tick_fs)
    ax.tick_params(length=0)
    for spine in ax.spines.values():
        spine.set_color("#cbd5e1")

A_main = attn_out.attentions[2][0, 2].numpy()
fig, ax = plt.subplots(figsize=(5.6, 4.8))
draw_attention(ax, A_main, attn_tokens)
ax.set_xlabel("token attended to (key)")
ax.set_ylabel("token being updated (query)")
fig.tight_layout()
plt.show()

> Setup the book does not print. Later cells depend on the state it creates, so run it.

In [ ]:
heads = [
    (4, 11, "previous-token head"),
    (5, 1, "attention sink"),
    (2, 2, "association head"),
]

fig, axes = plt.subplots(1, 3, figsize=(7.0, 2.7))
for k, (ax, (L, H, title)) in enumerate(zip(axes, heads)):
    A = attn_out.attentions[L][0, H].numpy()
    draw_attention(ax, A, attn_tokens,
                   annotate=False, tick_fs=7)
    ax.set_title(f"{title}\n(layer {L}, head {H})",
                 fontsize=8.5)
    if k > 0:
        ax.set_yticklabels([])
fig.tight_layout()
plt.show()

### Creating and using a fine-tuned model

`lst-openai-finetune`

> The book does not execute this listing. It needs values or a service you must supply yourself, so running it as printed will not work unedited.

In [ ]:
training = client.files.create(
    file=open("examples.jsonl", "rb"),
    purpose="fine-tune",
)

job = client.fine_tuning.jobs.create(
    training_file=training.id,
    model="<base-model>",
)

# once the job reports "succeeded"
reply = client.chat.completions.create(
    model=job.fine_tuned_model,
    messages=[{"role": "user", "content": "..."}],
)

> Setup the book does not print. Later cells depend on the state it creates, so run it.

In [ ]:
from transformers import AutoModelForCausalLM

dec_lm = AutoModelForCausalLM.from_pretrained("gpt2")
dec_lm.eval()

dec_prefix = "The bank raised interest"
dec_ids = attn_tok(dec_prefix, return_tensors="pt")
with torch.no_grad():
    dec_logits = dec_lm(**dec_ids).logits[0, -1]

> Setup the book does not print. Later cells depend on the state it creates, so run it.

In [ ]:
from matplotlib.patches import Rectangle

dec_panels = [(0.4, 0), (1.0, 0), (1.8, 2)]
fig, axes = plt.subplots(
    3, 2, figsize=(7.2, 7.0),
    gridspec_kw={"width_ratios": [1.55, 1.0]},
)

for (T, seed), (axb, axt) in zip(dec_panels, axes):
    probs = torch.softmax(dec_logits / T, dim=-1)
    top = torch.topk(probs, 6)
    toks = [attn_tok.decode([i]).strip() for i in top.indices]
    vals = top.values.numpy()
    sorted_p, _ = torch.sort(probs, descending=True)
    k = int((torch.cumsum(sorted_p, 0) < 0.9).sum()) + 1
    g = torch.Generator().manual_seed(seed)
    s_id = torch.multinomial(probs, 1, generator=g).item()
    s_tok = attn_tok.decode([s_id]).strip()
    in_top = s_tok in toks

    if in_top:
        ys = list(np.arange(len(toks))[::-1] + 1.6)
        show_toks, show_vals = toks, list(vals)
    else:
        ys = list(np.arange(len(toks))[::-1] + 2.4) + [0.5]
        show_toks = toks + [s_tok]
        show_vals = list(vals) + [float(probs[s_id])]
        axb.text(0.02, 1.42, "$\\vdots$",
                 fontsize=11, color="#94a3b8")

    colors = []
    for r in range(len(show_toks)):
        if r < min(k, 6) or (
            not in_top and r == len(show_toks) - 1 and k > 6
        ):
            colors.append("#1d4e89")
        else:
            colors.append("#cbd5e1")
    bars = axb.barh(ys, show_vals, color=colors, height=0.8)

    top_y = ys[0] + 0.55
    bot_y = (
        ys[min(k, 6) - 1] - 0.55 if k <= 6 else min(ys) - 0.55
    )
    axb.add_patch(Rectangle(
        (-0.01, bot_y), 1.06, top_y - bot_y,
        facecolor="#dceaf7", edgecolor="#1d4e89",
        linewidth=1.0, alpha=0.45, zorder=0,
    ))
    ntext = f"nucleus (p = 0.9): {k:,} token" + (
        "s" if k > 1 else ""
    )
    if k > 6:
        ntext += "\n(6 of them shown)"
        axb.text(1.02, top_y - 0.15, ntext, fontsize=7.5,
                 color="#1d4e89", ha="right", va="top")
    else:
        axb.text(1.02, bot_y - 0.25, ntext, fontsize=7.5,
                 color="#1d4e89", ha="right", va="top")

    s_idx = show_toks.index(s_tok)
    bars[s_idx].set_edgecolor("#b45309")
    bars[s_idx].set_linewidth(1.8)
    if show_vals[s_idx] > 0.6:
        axb.annotate(
            "sampled", xy=(show_vals[s_idx] - 0.02, ys[s_idx]),
            fontsize=8, color="#ffffff", va="center",
            ha="right", fontweight="bold",
        )
    else:
        axb.annotate(
            "sampled",
            xy=(max(show_vals[s_idx], 0.03) + 0.02, ys[s_idx]),
            fontsize=8, color="#b45309", va="center",
            fontweight="bold",
        )

    axb.set_yticks(ys)
    axb.set_yticklabels(show_toks, fontsize=8)
    axb.set_xlim(0, 1.05)
    axb.set_ylim(min(ys) - 0.9, max(ys) + 0.9)
    axb.set_title(f"temperature {T}", fontsize=10, loc="left",
                  fontweight="bold", color="#334155")
    axb.tick_params(labelsize=7.5, length=0)
    for sp in ["top", "right"]:
        axb.spines[sp].set_visible(False)
    for sp in ["left", "bottom"]:
        axb.spines[sp].set_color("#cbd5e1")

    axt.axis("off")
    axt.text(0.04, 0.78, "sampled token",
             fontsize=8, color="#64748b")
    axt.text(0.04, 0.56, f'"{s_tok}"', fontsize=13,
             color="#b45309", fontweight="bold")
    axt.text(0.04, 0.34, "appended, becomes the next prefix:",
             fontsize=8, color="#64748b")
    axt.text(0.04, 0.14,
             f"The bank raised interest {s_tok} ...",
             fontsize=8.5, color="#334155")

fig.tight_layout(h_pad=2.0)
plt.show()

### Initializing the API client from environment variables

`lst-openai-clients`

In [ ]:
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

client = OpenAI()

CHAT_MODEL = os.environ["CHAT_MODEL"]
EMBED_MODEL = os.environ["EMBED_MODEL"]

### A first chat completion request with system and user messages

`lst-first-chat-call`

In [ ]:
resp = client.chat.completions.create(
    model=CHAT_MODEL,
    messages=[
        {
            "role": "system",
            "content": (
                "You are a concise "
                "teaching assistant."
            ),
        },
        {
            "role": "user",
            "content": (
                "Explain in one sentence what a "
                "Large Language Model is."
            ),
        },
    ],
    temperature=0.2,
    max_tokens=60,
)

In [ ]:
print(resp.choices[0].message.content)

In [ ]:
print("Tokens:", resp.usage.total_tokens)

In [ ]:
import json

full = resp.model_dump()
full.pop("prompt_filter_results", None)
for c in full["choices"]:
    c.pop("content_filter_results", None)
full["usage"] = {
    k: v for k, v in full["usage"].items()
    if not isinstance(v, dict)
}
print(json.dumps(full, indent=2))

In [ ]:
params_example = {
    "n": 1,
    "temperature": 0.2,
    "max_tokens": 150,
    "top_p": 1.0,
    "frequency_penalty": 0.0,
    "presence_penalty": 0.0,
    "stop": None,
    "seed": 42,
}
print(params_example)

### Requesting JSON output and validating it against a simple contract

`lst-json-validation`

In [ ]:
task = (
    "Return valid JSON with keys: benefits, "
    "risks, summary. Each list must have "
    "exactly 3 short items."
)
resp = client.chat.completions.create(
    model=CHAT_MODEL,
    messages=[
        {
            "role": "system",
            "content": (
                "You are precise. Always return "
                "valid, minified JSON."
            ),
        },
        {
            "role": "user",
            "content": (
                "Benefits and risks of LLMs in "
                "higher education (3 each). "
                "Include a one-sentence summary."
            ),
        },
    ],
    temperature=0.2,
    max_tokens=250,
)

obj = json.loads(resp.choices[0].message.content)
assert set(obj.keys()) == {"benefits", "risks", "summary"}
assert len(obj["benefits"]) == 3
assert len(obj["risks"]) == 3
print(json.dumps(obj, indent=2))

In [ ]:
resp1 = client.chat.completions.create(
    model=CHAT_MODEL,
    messages=[
        {"role": "system", "content": "Be concise."},
        {
            "role": "user",
            "content": (
                "Propose a catchy title "
                "for a lecture on LLMs."
            ),
        },
    ],
    temperature=0.2,
    max_tokens=60,
)
print("CALL 1:", resp1.choices[0].message.content)

In [ ]:
resp2 = client.chat.completions.create(
    model=CHAT_MODEL,
    messages=[
        {"role": "system", "content": "Be concise."},
        {
            "role": "user",
            "content": (
                "What was the title "
                "I gave you earlier?"
            ),
        },
    ],
    temperature=0.2,
    max_tokens=60,
)
call2 = resp2.choices[0].message.content
print("CALL 2 (no history):", call2)

In [ ]:
messages = [
    {"role": "system", "content": "Be concise."},
    {
        "role": "user",
        "content": (
            "Propose a catchy title "
            "for a lecture on LLMs."
        ),
    },
]

resp_a = client.chat.completions.create(
    model=CHAT_MODEL,
    messages=messages,
    temperature=0.2,
    max_tokens=60,
)
assistant_msg = resp_a.choices[0].message
print("A:", assistant_msg.content)

In [ ]:
messages.append(
    {"role": "assistant", "content": assistant_msg.content}
)
messages.append(
    {
        "role": "user",
        "content": (
            "Great. Please shorten "
            "that title to five words."
        ),
    }
)
print(messages)

In [ ]:
resp_b = client.chat.completions.create(
    model=CHAT_MODEL,
    messages=messages,
    temperature=0.2,
    max_tokens=60,
)
print("B:", resp_b.choices[0].message.content)